# 📊 Notebook 3: Evaluation

Compare YOLO+CRNN vs TrOCR vs EasyOCR.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import time
import torch
import easyocr
import matplotlib.pyplot as plt
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

from src.model import CRNN
from src.utils import OCRPipeline, VOCAB_SIZE

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load our model (YOLO + CRNN)
crnn = CRNN(vocab_size=VOCAB_SIZE)
crnn.load_state_dict(torch.load("../crnn.pt", map_location=device))

our_pipeline = OCRPipeline(
    yolo_path="../runs/detect/train/weights/best.pt",
    crnn_model=crnn,
    device=device
)
print("✓ YOLO + CRNN loaded")

In [ ]:
# Load EasyOCR
easy_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("✓ EasyOCR loaded")

In [ ]:
# Test on sample image
import os
import cv2

# Get a test image
test_dir = "../datasets/SceneTrialTrain"
test_img = None
for f in os.listdir(test_dir):
    if f.endswith('.JPG'):
        test_img = os.path.join(test_dir, f)
        break

print(f"Testing on: {test_img}")

In [ ]:
# Compare inference times
print("\n" + "="*50)
print("INFERENCE COMPARISON")
print("="*50)

# Our model
start = time.time()
our_results = our_pipeline.predict(test_img)
our_time = time.time() - start
print(f"\nYOLO + CRNN: {our_time:.3f}s, {len(our_results)} detections")
for r in our_results:
    print(f"  → {r['text']} ({r['confidence']:.2f})")

# EasyOCR
start = time.time()
easy_results = easy_reader.readtext(test_img)
easy_time = time.time() - start
print(f"\nEasyOCR: {easy_time:.3f}s, {len(easy_results)} detections")
for bbox, text, conf in easy_results[:5]:
    print(f"  → {text} ({conf:.2f})")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Our model
img1 = cv2.imread(test_img)
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
for r in our_results:
    x, y, w, h = r['bbox']
    cv2.rectangle(img1, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(img1, r['text'], (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
axes[0].imshow(img1)
axes[0].set_title(f"YOLO + CRNN ({our_time:.2f}s)", fontweight='bold')
axes[0].axis('off')

# EasyOCR
img2 = cv2.imread(test_img)
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
for bbox, text, conf in easy_results:
    pts = [[int(p[0]), int(p[1])] for p in bbox]
    cv2.rectangle(img2, tuple(pts[0]), tuple(pts[2]), (255, 0, 0), 2)
    cv2.putText(img2, text, tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
axes[1].imshow(img2)
axes[1].set_title(f"EasyOCR ({easy_time:.2f}s)", fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"")
print(f"| Model        | Speed    | Detections |")
print(f"|--------------|----------|------------|")
print(f"| YOLO + CRNN  | {our_time:.3f}s   | {len(our_results):10} |")
print(f"| EasyOCR      | {easy_time:.3f}s   | {len(easy_results):10} |")